## Librerias

In [ ]:
import re
import spacy
import random
from spacy.training import Example
from spacy.training import offsets_to_biluo_tags
from spacy.util import minibatch, compounding
from typing import Optional
import logging
import os
import json
import csv
import unicodedata

In [2]:
# Configura el logging al principio del script (o en un archivo de configuración separado)
logging.basicConfig(level=logging.INFO,  # Nivel de detalle (INFO, DEBUG, WARNING, ERROR, CRITICAL)
                    format='%(asctime)s - %(levelname)s - %(message)s')

## Resource

In [ ]:
MONTH_MAP = {
    "ENERO": "01", "FEBRERO": "02", "MARZO": "03", "ABRIL": "04",
    "MAYO": "05", "JUNIO": "06", "JULIO": "07", "AGOSTO": "08",
    "SEPTIEMBRE": "09", "OCTUBRE": "10", "NOVIEMBRE": "11", "DICIEMBRE": "12"
}

## function to clean the text

In [ ]:
def ocr_clean(text: str) -> str:
    """
    Limpia texto OCR de manera optimizada:
    - Normaliza Unicode (NFKC).
    - Une palabras divididas por guiones.
    - Limpia caracteres especiales.
    - Elimina espacios innecesarios.
    """

    # 1. Normalizar caracteres Unicode
    text = unicodedata.normalize("NFKC", text)

    # 2. Unir palabras separadas por guiones al final de línea o espacios
    text = re.sub(r"(\w+)-\s*\n?\s*(\w+)", r"\1\2", text)

    # 3. Reemplazar cualquier salto de línea o tabulación por un único espacio
    text = re.sub(r"[\n\t]+", " ", text)

    # 4. Remover caracteres no deseados (excepto alfabeto español, números, puntuación común)
    text = re.sub(r"[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ().,\s-]", "", text)

    # 5. Normalizar signos de puntuación múltiples (; , :) a un punto único
    text = re.sub(r"[;,:]+", ".", text)

    # 6. Remover múltiples espacios seguidos
    text = re.sub(r" {2,}", " ", text)

    # 7. Remover espacios iniciales y finales
    return text.strip()


## Regex with characteres trash in the day

In [ ]:
def extraer_dia_mes(texto: str) -> Optional[tuple]:
    """
    Extrae día y mes de textos como:
    - (1.° DE MAYO)
    - (3 DE JUNIO)
    - 1 DE MAYO
    Retorna una tupla (día, mes) si tiene éxito, None en caso contrario.
    """

    texto = texto.upper()
    patron = re.search(r"\(?\s*(\d{1,2})[\.\°\"]*\s+DE\s+([A-ZÁÉÍÓÚ]+)\s*\)?", texto)

    if not patron:
        logging.warning(f"No se pudo extraer día/mes: '{texto}'")
        return None

    dia = patron.group(1).zfill(2)
    mes = MONTH_MAP.get(patron.group(2))

    if not mes:
        logging.warning(f"Mes desconocido: '{patron.group(2)}'")
        return None

    return dia, mes
    
def normalize_date_bip(texto: str, year="1848") -> Optional[str]:
    resultado = extraer_dia_mes(texto)
    if resultado:
        dia, mes = resultado
        return f"{dia}/{mes}/{year}"
    return None

def normalize_nuevo_fecha(spacy_fecha: str, year="1848") -> Optional[str]:
    resultado = extraer_dia_mes(spacy_fecha)
    if resultado:
        dia, mes = resultado
        return f"{dia}/{mes}/{year}"
    return None


## extract formula

In [ ]:
def extraer_formula(text: str, start_index: int) -> Optional[str]:
    """
    Dado 'text' y la posición donde termina el epígrafe,
    busca patrones de la fórmula (El Senado..., El Congreso...) 
    desde start_index hasta el final.
    Retorna el texto de la fórmula o None si no encuentra.
    """
    subtexto = text[start_index:]
    patrones_formulas = [
        r"El Senado y la Camara Representantes reunidos en Congreso",
        r"(?:El\s+Congreso\s+de\s+(?:la\s+)?Confederacion\s+Granadina",
        r"El\s+Senado\s+y\s+la\s+Camara\s+de\s+Representantes\s+(?:de\s+la\s+Nueva\s+Granada,\s*reunidos\s+en\s+Congreso)?"
        r"El Senado y la Camara de Representantes de la Nueva Granada,\s*reunidos en Congreso",
        r"El Senado y la Càmara de Representantes de la Nueva Granada,\s*reunidos en Congreso,",
        r"la Camara de Representantes de la Nueva Granada,\s*El Senado y\s*reunidos en Congreso,?",
        r"El Congreso de là Confederacion Granadina",
        r"EL CONGRESO DE LOS ESTADOS UNIDOS DE COLOMBIA",
        r"El Congreso de los Estados Unidos de Colombia"
    ]
     # Combina en un pattern alternado, usando re.DOTALL si quieres multiline
    pattern = "|".join(patrones_formulas)

    m = re.search(pattern, subtexto, re.IGNORECASE | re.DOTALL)
    if m:
        # match.group(1) es la parte capturada 
        formula_text = m.group(1).strip()
        return formula_text
    else:
        return None

## date

In [ ]:
def normalize_nuevo_fecha(spacy_fecha: str) -> Optional[str]:
    """
    Extrae el día y el mes de algo como "(3 DE JUNIO)" o "3 DE JUNIO"
    y retorna DD/MM/1448.
    """
    # Subimos a mayúsculas para la búsqueda
    text_upper = spacy_fecha.upper()

    # Regex simple: (\d{1,2})\s+DE\s+([A-Z]+)
    # O con paréntesis: \(\s*(\d{1,2})\s+DE\s+([A-Z]+)\s*\)
    patron = re.search(r"\(?\s*(\d{1,2})\s+DE\s+([A-ZÁÉÍÓÚ]+)\)?", text_upper)
    if not patron:
        logging.warning(f"No se pudo normalizar la fecha: '{spacy_fecha}'")
        return None

    day = patron.group(1).zfill(2)
    raw_mes = patron.group(2)

    month_map = {
        "ENERO": "01", "FEBRERO": "02", "MARZO": "03", "ABRIL": "04",
        "MAYO": "05", "JUNIO": "06", "JULIO": "07", "AGOSTO": "08",
        "SEPTIEMBRE": "09", "OCTUBRE": "10", "NOVIEMBRE": "11", "DICIEMBRE": "12"
    }
    mes_num = month_map.get(raw_mes, None)
    if not mes_num:
        logging.warning(f"Mes desconocido: '{raw_mes}' en fecha '{spacy_fecha}'")
        return None

    # El año lo forzamos a 1448
    year = "1848"
    return f"{day}/{mes_num}/{year}"

## LOAD MODEL AND TRAIN

In [ ]:
def load_train_data(input_file="train_old.json"):
    """
    Carga datos de entrenamiento en formato JSON.
    """
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            train_data = json.load(f)
            return train_data
    except FileNotFoundError:
        logging.error(f"Archivo de datos de entrenamiento no encontrado: {input_file}.")
        raise
    except json.JSONDecodeError:
        logging.error(f"Error al decodificar el archivo JSON: {input_file}.")
        raise
    except Exception as e:
        logging.error(f"Error inesperado al cargar TRAIN_DATA: {e}.")
        raise

In [ ]:
def anotar_archivos_de_carpeta(carpeta_txt: str):
    """
    Lee archivos .txt de una carpeta, limpia el texto, y anota las entidades.
    Retorna una lista con (texto_limpio, {"entities": [...]})
    """
    nuevos_ejemplos = []

    for archivo in os.listdir(carpeta_txt):
        if archivo.lower().endswith(".txt"):
            ruta = os.path.join(carpeta_txt, archivo)
            with open(ruta, "r", encoding="utf-8") as f:
                raw_text = f.read()

            # (A) Limpieza OCR
            cleaned_text = ocr_clean(raw_text)
            print(f"\n--- Texto limpio de {archivo} ---\n{cleaned_text}\nLongitud: {len(cleaned_text)}")

            # (B) Anotación manual (posiciones start/end + etiqueta)
            entidades = []
            while True:
                start = input("Inicio de entidad (o -1 para terminar): ")
                if start == "-1":
                    break
                end = input("Fin de entidad: ")
                label = input("Etiqueta: ")
                try:
                    start_idx = int(start)
                    end_idx = int(end)
                    entidades.append((start_idx, end_idx, label))
                except ValueError:
                    print("¡Índices inválidos! Intenta de nuevo.")

            # (C) Agregar a la lista
            nuevos_ejemplos.append((cleaned_text, {"entities": entidades}))

    return nuevos_ejemplos

## Extraction principall (spacy + regex)

In [ ]:
def spacy_extract_nuevo(text: str, nlp) -> dict:
    max_chars = 1500000
    if len(text) > max_chars:
        text = text[:max_chars]

    # -------------- IMPLEMENTACIÓN DEL NUEVO REGEX AQUÍ --------------
    patron_final = re.compile(
        r"(.*?)(?:\b(?:DECRETA|DECRET, AN|DECETA|CONSIDER ANDO|CONSIDERANDO)\b)",
        re.IGNORECASE | re.DOTALL
    )

    match_final = patron_final.search(text)

    if match_final:
        # Extraer el texto hasta la palabra clave encontrada
        fragment = match_final.group(1)
    else:
        # Si no encuentra la palabra clave, toma un fragmento predeterminado
        fragment = text[:400]

    fragment_clean = ocr_clean(fragment)

    doc = nlp(fragment_clean)

    result = {
        "fecha": None,
        "epigrafe": None,
        "numero_ley": None,
        "formula": None
    }

    epigrafe_start_end = None

    for ent in doc.ents:
        texto_entidad = ent.text.strip()
        if ent.label_ == "FECHA":
            fecha_norm = normalize_nuevo_fecha(texto_entidad)
            result["fecha"] = fecha_norm if fecha_norm else texto_entidad

        elif ent.label_ == "EPIGRAFE":
            result["epigrafe"] = texto_entidad
            epigrafe_start_end = (ent.start_char, ent.end_char)

        elif ent.label_ == "NUMERO_LEY":
            match_num = re.search(r"\bLE[YI]\s*\(\s*(\d+)\s*\)", texto_entidad, re.IGNORECASE)
            if match_num:
                result["numero_ley"] = match_num.group(1)

        elif ent.label_ == "FORMULA":
            result["formula"] = texto_entidad

    # --------------- Fallback de FORMULA ---------------
    if not result["formula"] and epigrafe_start_end:
        start_epigrafe = epigrafe_start_end[1]
        possible_formula = extraer_formula(fragment_clean, start_epigrafe)
        if possible_formula:
            result["formula"] = possible_formula

    return result


## model

In [ ]:
def train_spacy_model(train_data, output_dir, n_iter=30, learning_rate=0.001, dropout_rate=0.2):
    """
    Entrena un modelo spaCy NER desde cero y lo guarda en output_dir.
    """
    nlp = spacy.blank("es")
    ner = nlp.add_pipe("ner")

    for _, annotations in train_data:
        for ent in annotations["entities"]:
            ner.add_label(ent[2])

    optimizer = nlp.begin_training()

    for itn in range(n_iter):
        random.shuffle(train_data)
        losses = {}
        for text, annotations in train_data:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            nlp.update([example], drop=dropout_rate, sgd=optimizer, losses=losses)
        print(f"Iteración {itn + 1} - Pérdidas: {losses}")

    nlp.to_disk(output_dir)
    print(f"Modelo guardado en: {output_dir}")
    return nlp

In [ ]:
# Carga el dataset de entrenamiento
TRAIN_DATA = load_train_data("train_old.json")

nlp = spacy.blank("es")



# Entrena y guarda el modelo nuevo
nlp = train_spacy_model(
    TRAIN_DATA,
    output_dir="modelo_leyes_antiguas", 
    n_iter=30,
    learning_rate=0.001,
    dropout_rate=0.2
)

In [ ]:
TRAIN_DATA = load_train_data("train_old.json")
print("Cantidad de ejemplos de entrenamiento:", len(TRAIN_DATA))

## use

In [ ]:
def procesar_txt_en_carpeta(carpeta_txt: str, carpeta_salida: str, nlp, csv_output: str = None) -> None:
    os.makedirs(carpeta_salida, exist_ok=True)
    resultados = []

    for archivo in os.listdir(carpeta_txt):
        if archivo.lower().endswith(".txt"):
            ruta_txt = os.path.join(carpeta_txt, archivo)
            with open(ruta_txt, "r", encoding="utf-8") as f:
                texto = f.read()

            metadatos = spacy_extract_nuevo(texto, nlp)

            # Guardar JSON y CSV
            nombre_json = os.path.splitext(archivo)[0] + ".json"
            ruta_json = os.path.join(carpeta_salida, nombre_json)
            with open(ruta_json, "w", encoding="utf-8") as f:
                json.dump(metadatos, f, ensure_ascii=False, indent=4)
            print(f"Metadatos guardados en: {ruta_json}")

            resultados.append(metadatos)

    if csv_output and resultados:
        guardar_en_csv(resultados, csv_output)



def guardar_en_csv(resultados: list, nombre_csv: str) -> None:
    columnas = ["fecha", "epigrafe", "numero_ley", "formula"]  # Adaptado

    with open(nombre_csv, "w", encoding="utf-8", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas, delimiter=';')
        escritor.writeheader()

        for item in resultados:
            escritor.writerow({col: item.get(col, "") for col in columnas})

    print(f"Archivo CSV guardado: {nombre_csv}")


In [ ]:
if __name__ == "__main__":
    carpeta_txt = r"C:\Users\juans\Documents\project\Model-Extract-information\document\leyes\1854"
    carpeta_salida = r"C:\Users\juans\Documents\project\Model-Extract-information\document\salida_json"
    csv_salida = "metadatos_leyes_1844.csv"

    # Aquí debes cargar el MODEL0 spaCy, no llamar load_train_data
    nlp_model = spacy.load("modelo_leyes_antiguas")  # <-- ¡Correcto!

    procesar_txt_en_carpeta(carpeta_txt, carpeta_salida, nlp_model, csv_salida)